In [1]:
import arcpy

# --- Configuration ---
workDir = "C:/Users/mjrubino/OneDrive - State of North Carolina/PAD-US/"
arcpy.env.workspace = workDir + "PADUS4_1_StateNC.gdb"
fc = "USGSCatchment_PAD_Summary_copy"  # Path to your polygon feature class
id_field = "COMID"    # The field containing duplicate IDs
shape_field = "SHAPE@AREA"         # ArcPy token for area

# 1. Map IDs to the OID of the largest polygon
# Dictionary structure: { ID: (max_area, object_id) }
keepers = {}

with arcpy.da.SearchCursor(fc, ["OID@", id_field, shape_field]) as cursor:
    for row in cursor:
        oid, group_id, area = row
        
        # If ID is new or current area is larger than what we've seen for this ID
        if group_id not in keepers or area > keepers[group_id][0]:
            keepers[group_id] = (area, oid)

# 2. Extract just the OIDs we want to keep
keep_oids = {val[1] for val in keepers.values()}

# 3. Use UpdateCursor to delete anything NOT in the keep_oids set
with arcpy.da.UpdateCursor(fc, ["OID@"]) as cursor:
    count = 0
    for row in cursor:
        if row[0] not in keep_oids:
            cursor.deleteRow()
            count += 1

print(f"Deleted {count} duplicate polygons with smaller areas.")

Deleted 461 duplicate polygons with smaller areas.
